# Hyperparameter Tuning (Colab)

Mount Google Drive, install deps, and run `tune_model()` from the existing codebase.

In [ ]:
# ── Configuration ──
MODEL_NAME = 'grouped_mlp'  # 'xgboost', 'random_forest', 'mlp', 'grouped_mlp', or 'all'
N_TRIALS = 100
METRIC = 'log_loss'  # 'brier_score', 'log_loss', 'roc_auc', 'balanced_accuracy'
N_SPLITS = 5
RESAMPLE_METHOD = 'undersample'   # None, 'oversample', 'undersample', 'combined'

# Google Drive path containing models/ and turn_data.csv
DRIVE_PATH = '/content/drive/MyDrive/Colab Notebooks/vox-deorum'

In [ ]:
# ── Mount Google Drive & install dependencies ──
from google.colab import drive
drive.mount('/content/drive')

!pip install -q optuna xgboost lightgbm scikit-learn imbalanced-learn tabulate

In [ ]:
# ── Run hyperparameter tuning ──
import os, sys

models_dir = os.path.join(DRIVE_PATH, 'models')
os.chdir(models_dir)
# Add both models/ and analysis/ so nested imports resolve
sys.path.insert(0, models_dir)
sys.path.insert(0, DRIVE_PATH)

from tune_models import tune_model, SEARCH_SPACES

csv_path = os.path.join(DRIVE_PATH, 'turn_data.csv')
models_to_tune = list(SEARCH_SPACES.keys()) if MODEL_NAME == 'all' else [MODEL_NAME]

studies = {}
for name in models_to_tune:
    studies[name] = tune_model(
        model_name=name,
        n_trials=N_TRIALS,
        csv_path=csv_path,
        metric=METRIC,
        n_splits=N_SPLITS,
        resample_method=RESAMPLE_METHOD,
    )
    print()

## Tuning Visualization

In [ ]:
import matplotlib.pyplot as plt
from optuna.visualization.matplotlib import (
    plot_optimization_history,
    plot_param_importances,
    plot_parallel_coordinate,
    plot_slice,
)

for name, study in studies.items():
    print(f"\n{'='*60}")
    print(f"  {name.upper()}")
    print(f"{'='*60}\n")

    fig, ax = plt.subplots()
    plot_optimization_history(study, ax=ax)
    ax.set_title(f"{name} — Optimization History")
    plt.tight_layout(); plt.show()

    fig, ax = plt.subplots()
    plot_param_importances(study, ax=ax)
    ax.set_title(f"{name} — Parameter Importances")
    plt.tight_layout(); plt.show()

    plot_parallel_coordinate(study)
    plt.title(f"{name} — Parallel Coordinate")
    plt.tight_layout(); plt.show()

    plot_slice(study)
    plt.suptitle(f"{name} — Slice Plot", y=1.02)
    plt.tight_layout(); plt.show()